# Notebook 12 - Multimodal Grounding and Evaluation

This notebook turns multimodal evidence into measurable evaluation categories: answer quality, evidence quality, and localization quality.


## What you will learn

- how to score grounded multimodal answers
- how to separate answer failure from evidence failure
- how to organize multimodal error buckets


In [ ]:
!pip install -q numpy pandas matplotlib
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "12_multimodal_grounding_and_evaluation"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Create a grounded result table

The evaluation row should always link the answer to the evidence span that supports it.


In [ ]:
results = pd.DataFrame([
    {"example_id": "chart-1", "answer_correct": 1, "evidence_correct": 1, "localization_iou": 0.74},
    {"example_id": "invoice-1", "answer_correct": 1, "evidence_correct": 1, "localization_iou": 0.81},
    {"example_id": "audio-1", "answer_correct": 1, "evidence_correct": 0, "localization_iou": 0.00},
    {"example_id": "video-1", "answer_correct": 0, "evidence_correct": 1, "localization_iou": 0.53},
])
results


## Step 2: Derive grounded metrics

A grounded system must be right and cite the right place.


In [ ]:
results["grounded_success"] = ((results["answer_correct"] == 1) & (results["evidence_correct"] == 1)).astype(int)
metrics = pd.Series({
    "answer_accuracy": results["answer_correct"].mean(),
    "evidence_accuracy": results["evidence_correct"].mean(),
    "grounded_success": results["grounded_success"].mean(),
    "mean_iou": results["localization_iou"].mean(),
}).round(3)
metrics


## Step 3: Bucket the failures

Failure buckets turn notebook experiments into engineering priorities.


In [ ]:
def failure_bucket(row):
    if row.answer_correct == 0 and row.evidence_correct == 0:
        return "answer_and_evidence_failed"
    if row.answer_correct == 0:
        return "wrong_answer"
    if row.evidence_correct == 0:
        return "wrong_evidence"
    return "success"

results["failure_bucket"] = results.apply(failure_bucket, axis=1)
results[["example_id", "failure_bucket"]]


## Exercises and extensions

1. Add a modality column and summarize failure buckets by modality.
1. Create a stricter grounded_success metric that also requires localization_iou above a threshold.
